# Train and evaluate a model on the six folds

**How to use:** copy this notebook (e.g. `train_random_forest.ipynb`), set `MODEL_NAME` and `make_model()` in the settings cell, and run everything.
Results are saved to `results/`, so the team can compare every model in one table (last section).

- **The question:** for every I-5 northbound station in District 12, every 5 minutes, *will the station be congested (measured speed < 40 mph)
  at any point in the next 30 minutes?*
- **Features:** everyone uses `sf.SHARED_FEATURES` from `safeflow_features.py` (keep that file next to this notebook).
- **Validation:** six quarterly walk-forward folds, 2024 Q1 to 2025 Q2. Each trains on everything from January 2023 up to 7 days before the quarter.
  The July–December 2025 holdout is never read.
- **Main metric:** *AP, onsets*: average precision on stations not congested now (random guessing ≈ 0.05).

In [ ]:
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import safeflow_features as sf

pd.set_option("display.width", 160)

In [ ]:
# ---- Settings: change these ------------------------------------------------------------------------------------------
MODEL_NAME = "logistic_regression"          # used for the results file names, one name per model
QUICK_RUN = False                           # True = last fold only, for trying things out (not saved to the summary)
TRAIN_SAMPLE = 2_000_000                    # max training rows per fold (None = all rows, slower)
DATA_DIR = None                             # None = find the "Data" folder automatically


def make_model():
    # Return a new, unfitted model with fit() and predict_proba(). It is trained on df[sf.SHARED_FEATURES].
    # Notes on the features: some contain blanks (NaN), e.g. ramp flows and the typical-condition features. "dow" (day of week,
    # 0-4) is a category. Models that can't take NaN or need scaled inputs need preprocessing, as in this example.
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler

    numeric = [f for f in sf.SHARED_FEATURES if f != "dow"]
    prepare = ColumnTransformer([
        ("numeric", make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), numeric),
        ("categorical", OneHotEncoder(drop="first"), ["dow"]),
    ])
    return make_pipeline(prepare, LogisticRegression(max_iter=1000))     # <- replace with your model
# -------------------------------------------------------------------------------------------------------------------

## 1. Load the data

In [ ]:
df = sf.build_dataset(DATA_DIR)
FEATURES = sf.SHARED_FEATURES
print(f"{len(FEATURES)} shared features: {', '.join(FEATURES)}")

## 2. Train and score on each fold
For each fold: rebuild the fold-dependent features from that fold's training days, train on rows whose outcome was measured (`y_known`),
predict every validation row, and score.

In [ ]:
fold_results, predictions = [], []
for fold in sf.folds(df, only_last=QUICK_RUN):
    sf.add_typical_features(df, fold.train)
    train = df[fold.train & df["y_known"]]
    if TRAIN_SAMPLE and len(train) > TRAIN_SAMPLE:
        train = train.sample(TRAIN_SAMPLE, random_state=0)
    val = df[fold.val]

    model = make_model().fit(train[FEATURES], train["y"])
    probability = model.predict_proba(val[FEATURES])[:, 1]

    fold_results.append({"fold": fold.name, **sf.score(val, probability)})
    kept = val[sf.KEY + ["date", "y", "y_known", "congested_now", "real", "onset"]].copy()
    predictions.append(kept.assign(probability=probability))
    print(f"{fold.name}: AP onsets {fold_results[-1]['AP, onsets']:.3f} | caught ≥15 min {fold_results[-1]['caught ≥15 min']:.0%} | "
          f"false alarms/day {fold_results[-1]['false alarms / day']:.1f}")

fold_results = pd.DataFrame(fold_results).set_index("fold")
predictions = pd.concat(predictions, ignore_index=True).sort_values(sf.KEY).reset_index(drop=True)

## 3. Results per fold
Alert metrics use a threshold of 0.5 (alert when the probability is at least 50%). The spread across folds shows how stable the model is.

In [ ]:
table = fold_results.copy()
table.loc["mean"] = fold_results.mean()
table.loc["std"] = fold_results.std()
table.round(3)

## 4. Catching jams vs. false alarms, by threshold
All folds pooled. Lower thresholds catch more jams but raise more false alarms. The team picks the threshold from a false-alarm budget.

In [ ]:
thresholds = np.round(np.arange(0.1, 0.91, 0.1), 2)
tradeoff = pd.DataFrame([sf.alert_metrics(predictions, predictions["probability"].to_numpy(), t) for t in thresholds],
                        index=pd.Index(thresholds, name="threshold"))

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(tradeoff["false alarms / day"], tradeoff["caught ≥15 min"] * 100, marker="o", color="#2a78d6")
for t in (0.1, 0.5, 0.9):
    ax.annotate(f"{t}", (tradeoff.loc[t, "false alarms / day"], tradeoff.loc[t, "caught ≥15 min"] * 100),
                xytext=(5, 4), textcoords="offset points", fontsize=8, color="#52514e")
ax.set_xlabel("False-alarm episodes per day (whole corridor)")
ax.set_ylabel("% of onsets caught ≥15 min ahead")
ax.set_title(f"{MODEL_NAME}: warning trade-off", fontsize=11)
ax.grid(alpha=0.3)
plt.show()
tradeoff[["caught ≥15 min", "false alarms / day", "wrong-alert hours / day", "alerts right", "median lead (min)"]].round(3)

## 5. Are the probabilities honest? (calibration)
Predictions grouped by predicted probability, compared with how often congestion actually followed. A well-calibrated model sits on the diagonal.

In [ ]:
scored = predictions[predictions["y_known"]]
bins = pd.cut(scored["probability"], np.linspace(0, 1, 11), include_lowest=True)
calibration = scored.groupby(bins, observed=True).agg(predicted=("probability", "mean"), actual=("y", "mean"), rows=("y", "size"))
calibration = calibration[calibration["rows"] >= 100]

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.plot([0, 1], [0, 1], linestyle="--", color="#52514e", linewidth=1)
ax.plot(calibration["predicted"], calibration["actual"], marker="o", color="#2a78d6")
ax.set_xlabel("Predicted probability")
ax.set_ylabel("Actual share congested within 30 min")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title(f"{MODEL_NAME}: calibration", fontsize=11)
ax.grid(alpha=0.3)
plt.show()

## 6. Save and compare with the team
Saves this model's per-fold results to `results/<MODEL_NAME>_folds.csv` and its summary row to `results/summary.csv` (replacing any earlier row
with the same name). Quick runs aren't added to the summary. Commit the `results/` folder so everyone sees the same table.

In [ ]:
RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)

if QUICK_RUN:
    print("Quick run: results not saved.")
else:
    fold_results.round(4).to_csv(RESULTS / f"{MODEL_NAME}_folds.csv")
    row = {
        "model": MODEL_NAME,
        "date run": date.today().isoformat(),
        "features": len(FEATURES),
        "AP onsets (mean)": fold_results["AP, onsets"].mean(),
        "AP onsets (std)": fold_results["AP, onsets"].std(),
        "AP onsets (worst fold)": fold_results["AP, onsets"].min(),
        "Brier": fold_results["Brier"].mean(),
        "caught ≥15 min @0.5": fold_results["caught ≥15 min"].mean(),
        "false alarms/day @0.5": fold_results["false alarms / day"].mean(),
        "wrong-alert hours/day @0.5": fold_results["wrong-alert hours / day"].mean(),
    }
    summary_file = RESULTS / "summary.csv"
    summary = pd.read_csv(summary_file) if summary_file.exists() else pd.DataFrame()
    if len(summary):
        summary = summary[summary["model"] != MODEL_NAME]
    summary = pd.concat([summary, pd.DataFrame([row])], ignore_index=True).sort_values("AP onsets (mean)", ascending=False)
    summary.round(4).to_csv(summary_file, index=False)
    print(f"Saved {RESULTS / f'{MODEL_NAME}_folds.csv'} and updated {summary_file}")

pd.read_csv(RESULTS / "summary.csv").set_index("model").round(3) if (RESULTS / "summary.csv").exists() else None

## Adding a feature
Features are shared: add them in the **TEAM FEATURES** section at the bottom of `safeflow_features.py` (instructions and an example are there),
not in this notebook. A new feature joins `sf.SHARED_FEATURES` for everyone. Rerun your notebook to update your results.
Results saved before a feature change aren't directly comparable with results after it.